<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Breakout/Resistance_Breakout_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Resistance Breakout Strategy & Performance Scanner

This notebook implements a technical analysis scanner designed to identify **Resistance Breakouts**. The strategy focuses on identifying stocks that establish a clear price ceiling and subsequently break through it with high momentum.

**Key Features:**
* **Dynamic Resistance Detection:** Identifies price levels where a ticker has 'touched' or peaked multiple times (configurable via `min_touches`) within a lookback window.
* **Breakout Validation:** Triggers a signal only when the price closes above the resistance level by a specified percentage buffer and sets a new local high.
* **Ticker Locking:** Prevents redundant signals by enforcing a 'cool-off' period (e.g., 10 days) after a breakout is detected for a specific ticker.
* **Forward Performance Tracking:** Automatically calculates the 1-day through 5-day returns following each breakout and tracks the 'Success Rate' (percentage of time the price remains above the breakout level).
* **Buffered Data Ingestion:** Automatically downloads an extra 10 days of historical data beyond the scan window to ensure end-of-year signals have complete performance data.

**Current Configuration:** Scanning high-volume tickers for breakouts between January 2025 and December 2025.

In [49]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from datetime import timedelta

In [50]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: csv_input
Deleted DataFrame: full_historical_data
Deleted DataFrame: watchlist_df
Deleted DataFrame: df_filtered
Deleted DataFrame: results
Deleted DataFrame: performance_df
Deleted DataFrame: valid_subset
Deleted DataFrame: dia_df
All DataFrames cleared from memory.


In [51]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

### 1. Configuration and Data Ingestion
This cell defines the date range for the scan and calculates a 10-day 'buffer' for `DATA_END_DATE` to ensure we can track performance for year-end breakouts. It then loads the ticker list from Google Drive and downloads the necessary historical data via `yfinance`.

In [52]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

# 1. Parameterize scan range
# Get today's date for the end of the current scan period
TODAY_DATE = datetime.now().strftime('%Y-%m-%d')

# Data download should cover at least the lookback window (60 days) + scan window (30 days).
# Using 90 calendar days to be safe for 90 trading days.
START_DATA_DOWNLOAD_DT = pd.to_datetime(TODAY_DATE) - timedelta(days=180)
START_DATA_DOWNLOAD = START_DATA_DOWNLOAD_DT.strftime('%Y-%m-%d')

# Identify breakouts in the last 30 trading days (including today)
# Using 30 calendar days for simplicity, but in a real scenario, this might need a trading calendar.
START_BREAKOUT_IDENTIFICATION_DT = pd.to_datetime(TODAY_DATE) - timedelta(days=30)
START_BREAKOUT_IDENTIFICATION = START_BREAKOUT_IDENTIFICATION_DT.strftime('%Y-%m-%d')

# Print statements reflecting the ranges
print(f"Data download range: {START_DATA_DOWNLOAD} to {TODAY_DATE}")
print(f"Breakout scan range: {START_BREAKOUT_IDENTIFICATION} to {TODAY_DATE}")

# 2. Ingest list of tickers from your CSV file
tickers = []
try:
    csv_input = pd.read_csv(OptionVolume)
    tickers = csv_input['Symbol'].dropna().unique().tolist()
    tickers = [str(t).strip().upper() for t in tickers if str(t).isalpha()]
except Exception as e:
    print(f"Error reading CSV: {e}. Falling back to test list.")
    tickers = ["AAPL", "MSFT", "AMD", "NVDA", "INTC", "TSLA"]

# 3. Batch download historical data once using TODAY_DATE as the end date
full_historical_data = pd.DataFrame()
if tickers:
    print(f"Downloading historical data for {len(tickers)} tickers...")
    try:
        full_historical_data = yf.download(tickers, start=START_DATA_DOWNLOAD, end=TODAY_DATE, group_by='ticker', auto_adjust=True)
        print("Historical data download complete.")
    except Exception as e:
        print(f"Error fetching data: {e}")

if full_historical_data.empty:
    print("No historical data available for analysis.")

Data download range: 2025-12-20 to 2026-06-18
Breakout scan range: 2026-05-19 to 2026-06-18


[                       0%                       ]

[*********************100%***********************]  99 of 99 completed

Historical data download complete.


### 2. Resistance Breakout Scanner
This function iterates through the historical data to find price levels where a stock has peaked multiple times. It triggers a signal when the price decisively closes above that resistance level. It also includes a 'cool-off' mechanism to prevent redundant signals for the same stock.

In [53]:
def scan_resistance_breakout(ticker_list, full_data,
                             start_date_str, end_date_str,
                             resistance_buffer_pct=0.005, min_touches=3,
                             lookback_window=60, breakout_buffer_pct=0.0025,
                             cool_off_days=30):
    """
    Scans for unique resistance breakouts between start_date_str and end_date_str.
    """
    candidates = []
    last_breakout_date = {}

    # Convert strings to datetime objects for comparison
    start_dt = pd.to_datetime(start_date_str)
    end_dt = pd.to_datetime(end_date_str)

    print(f"Scanning {len(ticker_list)} tickers for triggers between {start_date_str} and {end_dt.strftime('%Y-%m-%d')}...")

    for ticker in ticker_list:
        try:
            if ticker not in full_data.columns.levels[0]:
                continue
            df = full_data[ticker].dropna().copy()
            if df.empty or len(df) < lookback_window + 2:
                continue

            last_breakout_date[ticker] = None

            for i in range(lookback_window, len(df)):
                current_date = df.index[i]

                # Filter: Only identify triggers within the defined scan window
                if current_date < start_dt or current_date > end_dt:
                    continue

                # Check if we are within the cool-off period
                if last_breakout_date[ticker] is not None:
                    days_since = (current_date - last_breakout_date[ticker]).days
                    if days_since < cool_off_days:
                        continue

                recent_df = df.iloc[i-lookback_window : i]
                current_close = df['Close'].iloc[i]
                previous_close = df['Close'].iloc[i-1]
                highest_high_in_window = recent_df['High'].max()

                # 1. Identify Resistance Levels
                highs = recent_df['High'].values
                sorted_highs = np.sort(highs)
                identified_levels = []

                for val in sorted_highs:
                    cluster = [h for h in highs if abs(h - val) / val <= resistance_buffer_pct]
                    if len(cluster) >= min_touches:
                        res_level = np.mean(cluster)
                        if not any(abs(res_level - lvl) / lvl < 0.02 for lvl, _ in identified_levels):
                            identified_levels.append((res_level, len(cluster)))

                # 2. Check Breakout Conditions
                for res_price, touches in identified_levels:
                    breakout_threshold = res_price * (1 + breakout_buffer_pct)

                    # Combined condition: Close above resistance AND new high for the lookback window
                    if (current_close > breakout_threshold) and \
                       (previous_close <= breakout_threshold) and \
                       (current_close > highest_high_in_window):
                            candidates.append({
                                "Ticker": ticker,
                                "Date": current_date.strftime('%Y-%m-%d'),
                                "Res_Level": round(res_price, 2),
                                "Breakout_Threshold": round(breakout_threshold, 2),
                                "Touches": touches,
                                "Close": round(current_close, 2),
                                "Prev_High": round(highest_high_in_window, 2)
                            })
                            last_breakout_date[ticker] = current_date
                            break

        except Exception:
            continue

    return pd.DataFrame(candidates)

# Re-run the scan using the parameter window
if 'full_historical_data' in globals() and not full_historical_data.empty:
    watchlist_df = scan_resistance_breakout(
        tickers,
        full_historical_data,
        start_date_str=START_BREAKOUT_IDENTIFICATION,
        end_date_str=TODAY_DATE,
        min_touches=5,
        cool_off_days=5
    )
    print(f"\nTotal triggers identified: {len(watchlist_df)}")
    display(watchlist_df.head(20))

Scanning 99 tickers for triggers between 2026-05-19 and 2026-06-18...

Total triggers identified: 11


,Ticker,Date,Res_Level,Breakout_Threshold,Touches,Close,Prev_High
0,MSFT,2026-05-29,428.69,429.76,5,450.24,432.76
1,AAPL,2026-06-02,312.04,312.82,5,315.20,315.00
2,IWM,2026-06-12,289.23,289.95,5,292.26,292.19
3,AVGO,2026-05-29,433.83,434.91,6,446.77,442.36
4,GS,2026-05-20,923.92,926.23,5,977.81,971.38
5,JPM,2026-06-12,315.65,316.44,13,320.72,320.24
6,SNOW,2026-05-28,177.77,178.22,5,239.20,184.74
7,C,2026-06-04,131.44,131.77,5,135.15,134.65
8,CSCO,2026-05-22,118.94,119.23,5,120.41,119.39
9,TGT,2026-06-11,129.67,129.99,5,132.64,131.85


### 3. Performance Analysis and Validation
This final section calculates the forward returns for 1 to 5 days following every identified breakout. It generates a summary table showing the average returns and a 'Success Rate,' verifying how often the price stayed above the breakout level.

In [54]:
def calculate_post_breakout_performance(watchlist, full_data):
    results = []

    for _, row in watchlist.iterrows():
        ticker = row['Ticker']
        breakout_date = row['Date']
        res_level = row['Res_Level']
        breakout_close = row['Close']

        if ticker not in full_data.columns.levels[0]:
            continue

        df = full_data[ticker].dropna()
        try:
            idx = df.index.get_loc(breakout_date)
        except KeyError:
            continue

        performance = row.to_dict()

        # Loop specifically for Days 1 through 5
        for d in [1, 2, 3, 4, 5]:
            future_idx = idx + d
            if future_idx < len(df):
                future_close = df['Close'].iloc[future_idx]
                ret = ((future_close - breakout_close) / breakout_close) * 100
                performance[f'Day_{d}_Return_%'] = round(ret, 2)
                performance[f'Day_{d}_Above_Res'] = bool(future_close >= res_level)
            else:
                performance[f'Day_{d}_Return_%'] = np.nan
                performance[f'Day_{d}_Above_Res'] = None

        results.append(performance)

    return pd.DataFrame(results)

if 'watchlist_df' in globals() and not watchlist_df.empty:
    performance_df = calculate_post_breakout_performance(watchlist_df, full_historical_data)

    # Calculate Summary for Days 1-5
    summary_data = {}
    validation_log = []

    for d in [1, 2, 3, 4, 5]:
        col_name = f'Day_{d}_Above_Res'
        valid_subset = performance_df[performance_df[col_name].notnull()]

        total_valid = len(valid_subset)
        success_count = valid_subset[col_name].sum() if total_valid > 0 else 0
        fail_count = total_valid - success_count

        avg_ret = valid_subset[f'Day_{d}_Return_%'].mean() if total_valid > 0 else np.nan
        pct_above = (success_count / total_valid * 100) if total_valid > 0 else 0

        summary_data[f'Day_{d} Avg Return %'] = avg_ret
        summary_data[f'Day_{d} % Above Res'] = pct_above

        validation_log.append({"Day": d, "Total": total_valid, "Above": success_count, "Below": fail_count})

    print("--- Updated Performance Summary (with 10-day buffer) ---")
    display(pd.Series(summary_data).to_frame(name='Value'))

    print("\n--- Raw Verification Counts ---")
    display(pd.DataFrame(validation_log))

    print("\n--- Detailed Performance Table (Full) ---")
    # Sort by Date in descending order before displaying
    performance_df['Date'] = pd.to_datetime(performance_df['Date'])
    display(performance_df.sort_values(by='Date', ascending=False))

--- Updated Performance Summary (with 10-day buffer) ---


,Value
Day_1 Avg Return %,0.999091
Day_1 % Above Res,81.818182
Day_2 Avg Return %,2.529091
Day_2 % Above Res,90.909091
Day_3 Avg Return %,0.758182
Day_3 % Above Res,63.636364
Day_4 Avg Return %,-1.727778
Day_4 % Above Res,55.555556
Day_5 Avg Return %,-2.127500
Day_5 % Above Res,62.500000



--- Raw Verification Counts ---


,Day,Total,Above,Below
0,1,11,9,2
1,2,11,10,1
2,3,11,7,4
3,4,9,5,4
4,5,8,5,3



--- Detailed Performance Table (Full) ---


,Ticker,Date,Res_Level,Breakout_Threshold,Touches,Close,Prev_High,Day_1_Return_%,Day_1_Above_Res,Day_2_Return_%,Day_2_Above_Res,Day_3_Return_%,Day_3_Above_Res,Day_4_Return_%,Day_4_Above_Res,Day_5_Return_%,Day_5_Above_Res
2,IWM,2026-06-12,289.23,289.95,5,292.26,292.19,0.81,True,-0.06,True,-0.81,True,NaN,None,NaN,None
5,JPM,2026-06-12,315.65,316.44,13,320.72,320.24,-0.41,True,3.25,True,3.97,True,NaN,None,NaN,None
9,TGT,2026-06-11,129.67,129.99,5,132.64,131.85,1.95,True,0.40,True,0.57,True,-3.64,False,NaN,None
7,C,2026-06-04,131.44,131.77,5,135.15,134.65,-1.98,True,-1.38,True,-0.31,True,-1.31,True,2.16,True
1,AAPL,2026-06-02,312.04,312.82,5,315.20,315.00,-1.57,False,-1.26,False,-2.49,False,-4.33,False,-7.82,False
10,ASML,2026-06-02,1648.28,1652.40,5,1705.37,1654.20,1.23,True,3.06,True,-3.73,False,2.56,True,4.25,True
0,MSFT,2026-05-29,428.69,429.76,5,450.24,432.76,2.28,True,-1.98,True,-5.09,False,-4.93,False,-7.46,False
3,AVGO,2026-05-29,433.83,434.91,6,446.77,442.36,2.95,True,7.79,True,7.27,True,-6.24,False,-13.66,False
6,SNOW,2026-05-28,177.77,178.22,5,239.20,184.74,6.84,True,17.12,True,9.17,True,0.87,True,2.08,True
8,CSCO,2026-05-22,118.94,119.23,5,120.41,119.39,-1.73,False,-0.61,True,-1.47,False,0.01,True,0.76,True
